# E11 on CLERC: band pipeline, Kaggle run

1. Builds the CLERC pool (`scripts/build_clerc_pool.py`): ~13 GB downloaded from Hugging Face, 10,000 full cases
   (gold + CLERC-negative cases of training queries), the queries, and a dev / test split by gold case.
2. Runs `scripts/run_band_pipeline.py --dataset clerc`, as pre-registered (E11, `results/notes/experiment_log.md`):
   smoke -> dev search -> one test run.
3. Packages the outputs.

**Input:** the private dataset `trr-code` (`trr_code.zip` from `kaggle/make_bundle.sh`). CLERC is downloaded by the notebook.

**Settings:** Internet **on** (Hugging Face download, pip). Accelerator: **none** (CPU; everything is lexical).

**Output:** `/kaggle/working/e11_clerc_outputs.zip`. Unzip it in the repository root.

In [ ]:
import glob, os, shutil, subprocess, time
from pathlib import Path

def find(pattern, archive):
    # a path inside an input dataset; if Kaggle kept the uploaded zip packed, unpack it to /tmp first
    for root in ("/kaggle/input", "/tmp/inputs"):
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
        if hits:
            return Path(hits[0])
        zips = sorted(glob.glob(f"/kaggle/input/**/{archive}", recursive=True))
        if root == "/kaggle/input" and zips:
            shutil.unpack_archive(zips[0], f"/tmp/inputs/{archive[:-4]}")
    raise FileNotFoundError(f"{pattern} not found under /kaggle/input: is the trr-code dataset added?")

code_root = find("configs/default.yaml", "trr_code.zip").parents[1]

REPO = Path("/kaggle/working/trr")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(code_root, REPO)
subprocess.run(["chmod", "-R", "u+w", str(REPO)], check=True)        # inputs are read-only
(REPO / "data/cache").mkdir(parents=True, exist_ok=True)
(REPO / "results/logs").mkdir(parents=True, exist_ok=True)
print("code at", REPO, "| git", (REPO / "GIT_HASH").read_text().strip())

DOWNLOAD = Path("/tmp/clerc_download")          # ~13 GB: outside /kaggle/working, so not saved as output
DOWNLOAD.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(DOWNLOAD).free / 1e9
print(f"free disk for the download: {free_gb:.0f} GB")
assert free_gb > 20, "need > 20 GB free under /tmp for the CLERC files"

In [ ]:
def run(cmd, log=None):
    """Run a shell command in the repo, stream its output, keep a log, stop the notebook on failure."""
    print("$", cmd, flush=True)
    t0, lines = time.time(), []
    with subprocess.Popen(cmd, shell=True, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            lines.append(line)
    if log:
        (REPO / "results/logs" / log).write_text("".join(lines))
    print(f"[{time.time() - t0:.0f} s]", flush=True)
    if p.returncode:
        raise RuntimeError(f"failed ({p.returncode}): {cmd}")

run("pip install -q -e '.[dev]'")
run("python -m pytest -q", log="e11_clerc_pytest.log")   # LegalBench real-data tests skip: no LegalBench here

## 1. Build the CLERC pool (download ~13 GB, stream the 1.84M-case collection once)

In [ ]:
run(f"python scripts/build_clerc_pool.py --download-dir {DOWNLOAD}", log="e11_clerc_pool.log")

## 2. Smoke run (50 cases): catches errors in a minute

In [ ]:
run("python scripts/run_band_pipeline.py --dataset clerc --stage dev --smoke 50", log="e11_clerc_smoke_dev.log")
run("python scripts/run_band_pipeline.py --dataset clerc --stage test --smoke 50", log="e11_clerc_smoke_test.log")

## 3. Dev search (dev queries only): picks the setting

In [ ]:
run("python scripts/run_band_pipeline.py --dataset clerc --stage dev", log="e11_clerc_dev.log")
print((REPO / "results/tables/band_dev_clerc.md").read_text())

## 4. Test run, once

In [ ]:
run("python scripts/run_band_pipeline.py --dataset clerc --stage test", log="e11_clerc_test.log")
print((REPO / "results/tables/band_test_clerc.md").read_text())

## 5. Package the outputs (tables, per-query runs, logs, and the pool itself for later runs)

In [ ]:
import zipfile
out = Path("/kaggle/working/e11_clerc_outputs.zip")
patterns = ["results/tables/band_*clerc*", "results/runs/*_test_clerc*.jsonl", "results/logs/e11_clerc_*.log",
            "data/external/clerc/pool_*/*"]
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in patterns:
        for f in sorted(REPO.glob(pattern)):
            z.write(f, f.relative_to(REPO))
            print(f.relative_to(REPO))
print(out, f"{out.stat().st_size / 1e6:.0f} MB")